# Quick Model Diagnostic

A lightweight version of the testing framework to quickly check feasibility and diagnostics.

In [1]:
import numpy as np
import pandas as pd
import time
import cvxpy as cp

from cma.data_reader import (
    read_vessel_class_data,
    read_port_data,
    read_sailing_distance_data,
    read_demand_with_transit_time,
    read_cnc_proforma_data,
)
from cma.port import PortGraph
from cma.servicegraph import ServiceGraph

np.set_printoptions(precision=4, suppress=True)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

## 1. Load Data (Minimal Subset)

In [2]:
vesselpool = read_vessel_class_data()
portpool_main, portpool_dmd = read_port_data()
dist_matrix = read_sailing_distance_data(portpool_main)
demand_matrix, transit_time_matrix = read_demand_with_transit_time(portpool_main)

portgraph = PortGraph(
    portpool_main, 
    dist_matrix, 
    demand_matrix,
    mat_transit_time=transit_time_matrix,
    filter_by_demand=False
)

proforma = read_cnc_proforma_data(portpool_main, vesselpool)
all_service_lines = proforma['lines']

# SUBSET FOR SPEED
service_lines = all_service_lines[:10]
servicegraph = ServiceGraph(service_lines)

trans_ports = portgraph.filtered_by_transship_capacity()
od_pairs_dict = servicegraph.get_all_paths(portgraph, trans_ports)

all_od_pairs = od_pairs_dict['od_pairs']
all_demands = od_pairs_dict['od_pairs_demand']
all_paths = od_pairs_dict['od_pairs_path']

filtered_pairs = []
filtered_paths = []
for od, paths, dmd in zip(all_od_pairs, all_paths, all_demands):
    if dmd > 0:
        filtered_pairs.append(od)
        filtered_paths.append(paths)

# SUBSET OD PAIRS
od_pairs = filtered_pairs[:50]
od_paths = filtered_paths[:50]

print(f"Loaded {len(service_lines)} lines and {len(od_pairs)} OD pairs for quick test.")

Loaded 10 lines and 50 OD pairs for quick test.


## 2. Run Optimization (All Features)

In [3]:
tuneparams = {
    'turnon-transship_shipclass_restriction': 1,
    'turnon-vessel_speed_optimization': 0,       # 0=ON (accurate)
    'turnon-port_operations_constraint': 1,
    'turnon-transit_time_penalty': 1,
    'ctrparam-kts_buffer': 0,
    'ctrparam-transship_A': 100,
    'ctrparam-speed_soft_cap_kts': 16.5,
    'ctrparam-speed_penalty_multiplier': 2.0,
    'ctrparam-transit_penalty_multiplier': 1000.0,
    'ctrparam-buffer_penalty_below_15pct': 1000.0,
    'ctrparam-buffer_penalty_above_30pct': 2000.0,
    'BigM-transship': 10000,
    'BigM-n_ships': 10,
    'BigM-saildays': 1000,
    'BigM-line_capacity': 100000,
    'BigM-portcall_cost': 1e7,          # Tightened from 2e9 for better stability
    'solver-MIPGap': 0.05,              # 5% gap (returns "good enough" results much faster)
    'solver-TimeLimit': 21600,          
    'solver-MIPFocus': 1,               # Focus on finding feasible solutions quickly
    'solver-verbose': True              # Show Gurobi's progress logs
}

week_levels = [1, 2, 3, 4, 5, 6, 7, 8]

start_time = time.time()
solution = servicegraph.fulfill_demands(
    od_pairs,
    od_paths,
    portgraph,
    vesselpool,
    week_levels,
    tuneparams
)
end_time = time.time()

print(f"Solve Time: {end_time - start_time:.2f}s")
print(f"Total Cost: {solution['total cost']}")

c:\Users\ASUS\.conda\envs\py311\Lib\site-packages\cvxpy\expressions\expression.py:683: UserWarning: 
This use of ``*`` has resulted in matrix multiplication.
Using ``*`` for matrix multiplication has been deprecated since CVXPY 1.1.
    Use ``*`` for matrix-scalar and vector-scalar multiplication.
    Use ``@`` for matrix-matrix and matrix-vector multiplication.
    Use ``multiply`` for elementwise multiplication.
This code path has been hit 1 times so far.

  warnings.warn(msg, UserWarning)
c:\Users\ASUS\.conda\envs\py311\Lib\site-packages\cvxpy\expressions\expression.py:683: UserWarning: 
This use of ``*`` has resulted in matrix multiplication.
Using ``*`` for matrix multiplication has been deprecated since CVXPY 1.1.
    Use ``*`` for matrix-scalar and vector-scalar multiplication.
    Use ``@`` for matrix-matrix and matrix-vector multiplication.
    Use ``multiply`` for elementwise multiplication.
This code path has been hit 2 times so far.

  warnings.warn(msg, UserWarning)
c:\Use

                                     CVXPY                                     
                                     v1.7.5                                    


(CVXPY) Feb 05 12:39:41 AM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Feb 05 12:39:41 AM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Feb 05 12:39:41 AM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Feb 05 12:39:41 AM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) Feb 05 12:39:41 AM: Compiling problem (target solver=GUROBI).
(CVXPY) Feb 05 12:39:41 AM: Reduction chain: CvxAttr2Constr -> Qp2SymbolicQp -> QpMatrixStuffing -> GUROBI
(CVXPY) Feb 05 12:39:41 AM: Applying reduction CvxAttr2Constr


-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------


(CVXPY) Feb 05 12:39:42 AM: Applying reduction Qp2SymbolicQp
(CVXPY) Feb 05 12:39:43 AM: Applying reduction QpMatrixStuffing
(CVXPY) Feb 05 12:39:47 AM: Applying reduction GUROBI
(CVXPY) Feb 05 12:39:47 AM: Finished problem compilation (took 6.082e+00 seconds).
(CVXPY) Feb 05 12:39:47 AM: Invoking solver GUROBI  to obtain a solution.


-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------
Set parameter Username
Set parameter LicenseID to value 2725074
Academic license - for non-commercial use only - expires 2026-10-20
Set parameter OutputFlag to value 1
Set parameter QCPDual to value 1
Set parameter MIPGap to value 0.05
Set parameter TimeLimit to value 21600
Set parameter MIPFocus to value 1
Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (win64 - Windows 11.0 (26100.2))

CPU model: AMD Ryzen 7 5700U with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Non-default parameters:
TimeLimit  21600
MIPGap  0.05
MIPFocus  1
QCPDual  1

Optimize a model with 15152 rows, 6896 columns and 116237 nonzeros
Model fingerprint: 0xb2178944
Variable types: 4706 continuous

(CVXPY) Feb 05 12:39:52 AM: Problem status: optimal
(CVXPY) Feb 05 12:39:52 AM: Optimal value: 2.084e+07
(CVXPY) Feb 05 12:39:52 AM: Compilation took 6.082e+00 seconds
(CVXPY) Feb 05 12:39:52 AM: Solver (including time spent in interface) took 5.152e+00 seconds


-------------------------------------------------------------------------------
                                    Summary                                    
-------------------------------------------------------------------------------
Solve Time: 13.24s
Total Cost: 20842609.31628199


## 3. Diagnostics & Soft Constraint Analysis

In [4]:
if solution['total cost'] == float('inf'):
    print("❌ STILL INFEASIBLE. Checking hard constraints...")
    # ... (similar check as in test_buffer_constraint.ipynb) ...
else:
    print("✓ FEASIBLE! Analyzing soft constraint violations:")
    
    # Analyze Buffer Violations
    violation_lb = solution['buffer violation lb']
    violation_ub = solution['buffer violation ub']
    
    violation_data = []
    for i, line in enumerate(service_lines):
        lb_val = violation_lb[i].value if hasattr(violation_lb[i], 'value') else 0
        ub_val = violation_ub[i].value if hasattr(violation_ub[i], 'value') else 0
        if lb_val > 0.01 or ub_val > 0.01:
            violation_data.append({
                'Line': line.name(),
                'Below 15% (h)': lb_val,
                'Above 30% (h)': ub_val
            })
    
    if violation_data:
        print("\nBuffer Violations Detected:")
        print(pd.DataFrame(violation_data))
    else:
        print("\nNo buffer violations! All lines within 15-30% range.")

    # Analyze Weeks/Vessels picked
    weeks_vars = solution['weeks']
    picked_weeks = []
    for i, line in enumerate(service_lines):
        for j, wk in enumerate(week_levels):
            if weeks_vars[i, j].value > 0.5:
                picked_weeks.append({'Line': line.name(), 'Week': wk})
    
    print("\nCycle Times (Weeks) Picked:")
    print(pd.DataFrame(picked_weeks))

✓ FEASIBLE! Analyzing soft constraint violations:

Buffer Violations Detected:
      Line  Below 15% (h)  Above 30% (h)
0  BBX3CNC            0.0      39.909736
1   CP3CNC            0.0       2.359413

Cycle Times (Weeks) Picked:
      Line  Week
0  BBX2CNC     3
1  BBX3CNC     3
2   BBXCNC     2
3   BMXCNC     2
4  CHN1CNC     3
5  CMS2CNC     3
6   CP2CNC     2
7   CP3CNC     2
8   CP8CNC     1
9   CS1CNC     3


## 4. Full Dataset Evaluation

Testing the model on all service lines and all positive demand OD pairs to identify potential bottlenecks.

In [5]:
service_lines_full = all_service_lines
servicegraph_full = ServiceGraph(service_lines_full)

print("Finding all paths for full dataset...")
trans_ports = portgraph.filtered_by_transship_capacity()
od_pairs_dict_full = servicegraph_full.get_all_paths(portgraph, trans_ports)

all_od_pairs_full = od_pairs_dict_full['od_pairs']
all_demands_full = od_pairs_dict_full['od_pairs_demand']
all_paths_full = od_pairs_dict_full['od_pairs_path']

filtered_pairs_full = []
filtered_paths_full = []
for od, paths, dmd in zip(all_od_pairs_full, all_paths_full, all_demands_full):
    if dmd > 0:
        filtered_pairs_full.append(od)
        filtered_paths_full.append(paths)

print(f"Full Dataset: {len(service_lines_full)} lines and {len(filtered_pairs_full)} OD pairs.")

start_time = time.time()
solution_full = servicegraph_full.fulfill_demands(
    filtered_pairs_full,
    filtered_paths_full,
    portgraph,
    vesselpool,
    week_levels,
    tuneparams
)
end_time = time.time()

print(f"Full Solve Time: {end_time - start_time:.2f}s")
print(f"Total Cost: {solution_full['total cost']}")

Finding all paths for full dataset...
Full Dataset: 31 lines and 741 OD pairs.


c:\Users\ASUS\.conda\envs\py311\Lib\site-packages\cvxpy\expressions\expression.py:683: UserWarning: 
This use of ``*`` has resulted in matrix multiplication.
Using ``*`` for matrix multiplication has been deprecated since CVXPY 1.1.
    Use ``*`` for matrix-scalar and vector-scalar multiplication.
    Use ``@`` for matrix-matrix and matrix-vector multiplication.
    Use ``multiply`` for elementwise multiplication.
This code path has been hit 11 times so far.

  warnings.warn(msg, UserWarning)
c:\Users\ASUS\.conda\envs\py311\Lib\site-packages\cvxpy\expressions\expression.py:683: UserWarning: 
This use of ``*`` has resulted in matrix multiplication.
Using ``*`` for matrix multiplication has been deprecated since CVXPY 1.1.
    Use ``*`` for matrix-scalar and vector-scalar multiplication.
    Use ``@`` for matrix-matrix and matrix-vector multiplication.
    Use ``multiply`` for elementwise multiplication.
This code path has been hit 12 times so far.

  warnings.warn(msg, UserWarning)
c:\U

                                     CVXPY                                     
                                     v1.7.5                                    


(CVXPY) Feb 05 12:40:02 AM: Your problem has 31890 variables, 47416 constraints, and 0 parameters.
(CVXPY) Feb 05 12:40:03 AM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Feb 05 12:40:03 AM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Feb 05 12:40:03 AM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Feb 05 12:40:03 AM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) Feb 05 12:40:05 AM: Compiling problem (target solver=GUROBI).
(CVXPY) Feb 05 12:40:05 AM: Reduction chain: CvxAttr2Constr -> Qp2SymbolicQp -> QpMatrixStuffing -> GUROBI
(CVXPY) Feb 05 12:40:05 AM: Applying reduction CvxAttr2Constr


-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------


(CVXPY) Feb 05 12:40:09 AM: Applying reduction Qp2SymbolicQp
(CVXPY) Feb 05 12:40:14 AM: Applying reduction QpMatrixStuffing
(CVXPY) Feb 05 12:43:44 AM: Applying reduction GUROBI
(CVXPY) Feb 05 12:43:44 AM: Finished problem compilation (took 2.210e+02 seconds).
(CVXPY) Feb 05 12:43:44 AM: Invoking solver GUROBI  to obtain a solution.


-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------
Set parameter OutputFlag to value 1
Set parameter QCPDual to value 1
Set parameter MIPGap to value 0.05
Set parameter TimeLimit to value 21600
Set parameter MIPFocus to value 1
Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (win64 - Windows 11.0 (26100.2))

CPU model: AMD Ryzen 7 5700U with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Non-default parameters:
TimeLimit  21600
MIPGap  0.05
MIPFocus  1
QCPDual  1

Optimize a model with 47416 rows, 31890 columns and 544503 nonzeros
Model fingerprint: 0xc1c17be4
Variable types: 25101 continuous, 6789 integer (6448 binary)
Coefficient statistics:
  Matrix range     [4e-05, 1e+07]
  Objective range  [1e-01, 1e+06]
  Bounds 

c:\Users\ASUS\.conda\envs\py311\Lib\site-packages\cvxpy\problems\problem.py:1539: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(
(CVXPY) Feb 05 06:43:47 AM: Problem status: user_limit
(CVXPY) Feb 05 06:43:47 AM: Optimal value: 4.306e+07
(CVXPY) Feb 05 06:43:47 AM: Compilation took 2.210e+02 seconds
(CVXPY) Feb 05 06:43:47 AM: Solver (including time spent in interface) took 2.160e+04 seconds


-------------------------------------------------------------------------------
                                    Summary                                    
-------------------------------------------------------------------------------
Full Solve Time: 21845.29s
Total Cost: 43058309.75913824


In [6]:
if solution_full['total cost'] == float('inf'):
    print("\n❌ FULL DATASET INFEASIBLE. Identifying problematic components...")
    
    # 1. Check for ports with zero productivity that have demand
    zero_prod_ports = []
    for port in portgraph.tolist_port():
        prods = port.get_producticity(vesselpool)
        if sum(prods) == 0:
            zero_prod_ports.append(port.get_id())
    
    if zero_prod_ports:
        print(f"\nPorts with ZERO productivity: {zero_prod_ports}")
        # Check if any OD pair involves these ports
        for i, (o_idx, d_idx) in enumerate(filtered_pairs_full):
            o_id = portgraph.get_port_by_idx(o_idx).get_id()
            d_id = portgraph.get_port_by_idx(d_idx).get_id()
            if o_id in zero_prod_ports or d_id in zero_prod_ports:
                print(f"  ⚠️ OD Pair {o_id}->{d_id} involves zero-productivity port.")

    # 2. Check for impossible distance/speed requirements (Hard Constraints)
    print("\nChecking for distance/speed violations:")
    for line in service_lines_full:
        dist = line.get_distance(portgraph)
        max_weeks = max(week_levels)
        min_speed_needed = dist / (24 * (7 * max_weeks - 1.0)) # 1 day min stay
        if min_speed_needed > 18.5:
            print(f"  ❌ Line {line.name()}: {dist:.0f}nm needs {min_speed_needed:.1f} kts at {max_weeks} weeks (Max 18.5)")

else:
    print("\n✓ FULL DATASET FEASIBLE!")
    
    # Violation Summary
    violation_lb = solution_full['buffer violation lb']
    violation_ub = solution_full['buffer violation ub']
    violation_summary = []
    for i, line in enumerate(service_lines_full):
        lb = violation_lb[i].value if hasattr(violation_lb[i], 'value') else 0
        ub = violation_ub[i].value if hasattr(violation_ub[i], 'value') else 0
        if lb > 0.1 or ub > 0.1:
            violation_summary.append({'Line': line.name(), 'LB_Violation': lb, 'UB_Violation': ub})
    
    if violation_summary:
        print("\nSignificant Buffer Violations in Full Dataset:")
        print(pd.DataFrame(violation_summary))


✓ FULL DATASET FEASIBLE!

Significant Buffer Violations in Full Dataset:
      Line  LB_Violation  UB_Violation
0  BBX3CNC           0.0     39.909736
1   BMXCNC           0.0     22.138156
2   CP3CNC           0.0      2.359413
3   SGSCNC           0.0      0.199653
